# 02 — Split, scaling and resampling (leakage-safe)
Rules: split first; RobustScaler on Time and Amount only; scaler and SMOTE live inside an imblearn Pipeline, so they are fitted on training folds only. The test set keeps its real imbalance.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config
from src.preprocess import load_data, make_preprocessor, train_test
from src.train import build_pipeline, cv_splitter

df = load_data(verbose=False)
X_train, X_test, y_train, y_test = train_test(df)

## Stratification check (SRS 11.2)

In [2]:
print(f"Fraud % overall: {df.Class.mean():.4%}")
print(f"Fraud % train:   {y_train.mean():.4%}")
print(f"Fraud % test:    {y_test.mean():.4%}")
assert abs(y_train.mean() - y_test.mean()) < 0.0002

Fraud % overall: 0.1667%
Fraud % train:   0.1665%
Fraud % test:    0.1674%


## Leakage check (SRS 11.2): no test row appears in training

In [3]:
overlap_idx = set(X_train.index) & set(X_test.index)
row_overlap = pd.merge(X_train, X_test, how="inner").shape[0]
print(f"Shared index labels: {len(overlap_idx)} | identical rows in both sets: {row_overlap}")
assert len(overlap_idx) == 0 and row_overlap == 0

Shared index labels: 0 | identical rows in both sets: 0


## Scaler is fitted on training data only

In [4]:
pre = make_preprocessor().fit(X_train)
scaler = pre.named_transformers_["robust"]
print("Scaler median (Time, Amount):", scaler.center_.round(2))
print("Train median  (Time, Amount):", X_train[config.SCALE_COLS].median().values.round(2))
print("Full-data median would be:   ", df[config.SCALE_COLS].median().values.round(2), "(not used)")

Xt = pd.DataFrame(pre.transform(X_test), columns=pre.get_feature_names_out())
print("\nScaled test Time/Amount summary:")
print(Xt[config.SCALE_COLS].describe().loc[["mean", "50%", "std"]].round(3))

Scaler median (Time, Amount): [8.4816e+04 2.2080e+01]
Train median  (Time, Amount): [8.4816e+04 2.2080e+01]
Full-data median would be:    [8.46925e+04 2.20000e+01] (not used)

Scaled test Time/Amount summary:
       Time  Amount
mean  0.113   0.925
50%  -0.004  -0.007
std   0.557   3.716


## Resampling only changes the training data inside `fit`
The pipeline is fitted on one training fold; SMOTE / undersampling happen there, and the validation fold is scored with its real imbalance.

In [5]:
pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
fold_train, fold_val = next(cv_splitter().split(X_train, y_train))
Xf, yf = X_train.iloc[fold_train], y_train.iloc[fold_train]

for tech in ["none", "undersample", "smote"]:
    pipe = build_pipeline("lr", tech, pos_weight)
    Xs = pipe.named_steps["scale"].fit_transform(Xf)
    sampler = pipe.named_steps["resample"]
    if sampler == "passthrough":
        ys = yf
    else:
        _, ys = sampler.fit_resample(Xs, yf)
    print(f"{tech:<12} rows seen by model: {len(ys):>7,} | fraud share {ys.mean():.2%}")

print(f"\nValidation fold (unchanged): {len(fold_val):,} rows, fraud share {y_train.iloc[fold_val].mean():.3%}")
print(f"Class weight for fraud (legit/fraud ratio): {pos_weight:.1f}")

none         rows seen by model: 181,584 | fraud share 0.17%


undersample  rows seen by model:     606 | fraud share 50.00%


  File "D:\Credit Card Fraud Detection\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Gulab Ahmad\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Gulab Ahmad\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Gulab Ahmad\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


smote        rows seen by model: 362,562 | fraud share 50.00%

Validation fold (unchanged): 45,396 rows, fraud share 0.165%
Class weight for fraud (legit/fraud ratio): 599.5
